# Feature Engineering

This notebook creates additional business-oriented features that enhance the analytical value of the original dataset. These new variables will be used throughout the business analysis and dashboard development.

In [19]:
import pandas as pd
import numpy as np

In [20]:
products = pd.read_csv("../data/product_info.csv")

## Dataset Overview

Load the cleaned dataset and verify its structure before creating new variables.

In [21]:
products.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


## Price Segmentation

Products are grouped into pricing tiers to simplify comparisons across different market segments.

In [22]:
def price_segment(price):
    if price < 20:
        return "Budget"
    elif price < 50:
        return "Mid-range"
    elif price < 100:
        return "Premium"
    else:
        return "Luxury"

products["price_segment"] = products["price_usd"].apply(price_segment)

In [23]:
products["price_segment"].value_counts()

price_segment
Mid-range    4724
Premium      1634
Budget       1205
Luxury        931
Name: count, dtype: int64

## Engagement Score

A custom engagement metric is created by combining customer interest (favorites) and review activity.

The logarithm reduces the influence of extremely popular products while preserving ranking differences.

In [24]:
products["engagement_score"] = (
    np.log1p(products["loves_count"]) +
    products["reviews"].fillna(0)
)

In [25]:
products[["engagement_score"]].describe()

,engagement_score
count,8494.000000
mean,442.973013
std,1087.456452
min,0.000000
25%,30.374597
50%,121.464117
75%,411.841608
max,21294.247384


## Rating Classification

Product ratings are grouped into qualitative levels for easier interpretation.

In [26]:
def rating_level(rating):
    if pd.isna(rating):
        return "No Rating"
    elif rating >= 4.5:
        return "Excellent"
    elif rating >= 4:
        return "Good"
    elif rating >= 3:
        return "Average"
    else:
        return "Poor"

products["rating_level"] = products["rating"].apply(rating_level)

In [27]:
products["rating_level"].value_counts()

rating_level
Good         3728
Excellent    2375
Average      1900
No Rating     278
Poor          213
Name: count, dtype: int64

## Price per Review

Estimate how much product price corresponds to each customer review.

Higher values may indicate products with limited customer engagement relative to their price.

In [28]:
products["price_per_review"] = (
    products["price_usd"] /
    (products["reviews"].fillna(0) + 1)
)

## Discount Indicator

Identify whether a product is currently offered at a discounted price.

In [29]:
products["has_discount"] = products["sale_price_usd"].notna()

In [30]:
products["has_discount"].value_counts()

has_discount
False    8224
True      270
Name: count, dtype: int64

## Export Engineered Dataset

Save the enriched dataset for the business analysis stage.

In [31]:
products.to_csv(
    "../data/products_engineered.csv",
    index=False
)

## Feature Engineering Summary

Several business-oriented features were created to enrich the original dataset and support more meaningful analysis.

These variables simplify comparisons, improve segmentation, and enable the identification of patterns that are not directly observable from the raw data alone.